In [ ]:
import json
import os
from pathlib import Path

import numpy as np

SOURCE_ROOT = Path('..')
DATASET_ROOT = Path('dataset')

# Set to a relative tok path string to process only one segment, or None for all.
TEST_ONLY_TOK = None

# Idempotent/resume behavior: skip segments already merged.
SKIP_ALREADY_PROCESSED = True


def map_tok_json_to_input_npy(tok_path: Path) -> Path:
    p = tok_path.as_posix()

    # Try the old colocated location first.
    colocated = p.replace('.align.tok.json', '.npy')

    # Map old scraped roots to new pooled-feature roots.
    mapped = (
        p.replace('digi/scraped', 'Digi24')
        .replace('prima/scraped', 'PrimaTV')
        .replace('protv/scraped', 'ProTV')
        .replace('.align.tok.json', '.npy')
    )

    colocated_path = Path(colocated)
    mapped_path = Path(mapped)

    if colocated_path.exists():
        return colocated_path
    return mapped_path


def to_global_frame_idx(value: float, n_frames: int) -> int:
    if n_frames <= 0:
        return 0

    v = float(value)
    if 0.0 <= v <= 1.0:
        idx = int(round(v * (n_frames - 1)))
    else:
        idx = int(round(v))

    return max(0, min(n_frames - 1, idx))


for root, _, files in os.walk(SOURCE_ROOT):
    tok_files = sorted(file for file in files if file.endswith('.align.tok.json'))
    if not tok_files:
        continue

    for file in tok_files:
        tok_path = Path(root) / file

        rel_tok_path = tok_path.relative_to(SOURCE_ROOT)
        if TEST_ONLY_TOK is not None and rel_tok_path != Path(TEST_ONLY_TOK):
            continue

        inp_npy = map_tok_json_to_input_npy(tok_path)
        if not inp_npy.exists():
            print(f'[MISSING NPY] {tok_path} -> {inp_npy}')
            continue

        rel_segment_path = rel_tok_path.with_suffix('').with_suffix('')
        out_prefix = DATASET_ROOT / rel_segment_path
        out_feat_path = Path(str(out_prefix) + '.npy')
        out_json_path = Path(str(out_prefix) + '.json')

        if SKIP_ALREADY_PROCESSED and out_feat_path.exists() and out_json_path.exists():
            print(f'[SKIP] {rel_segment_path}')
            continue

        feats = np.load(inp_npy)
        n_frames = int(feats.shape[0])

        with open(tok_path, 'r', encoding='utf-8') as fin:
            js = json.load(fin)

        segments = js.get('segments', [])
        merged_labels = []

        for i, segment in enumerate(segments):
            sent_id = str(segment.get('id', str(i).zfill(3)))

            text = segment.get('text', '')
            text_lower = segment.get('text_lower', text.lower())

            segment_tokens = segment.get('tokens', [])
            if isinstance(segment_tokens, dict):
                tokens = (
                    segment_tokens.get('input_ids')
                    or segment_tokens.get('ids')
                    or segment_tokens.get('token_ids')
                    or []
                )
            elif isinstance(segment_tokens, list):
                tokens = segment_tokens
            else:
                tokens = []

            tokens_lower = segment.get('tokens_lower', [])
            if not tokens_lower:
                if isinstance(segment_tokens, dict):
                    tokens_lower = (
                        segment_tokens.get('input_ids_lower')
                        or segment_tokens.get('ids_lower')
                        or segment_tokens.get('lower_input_ids')
                        or []
                    )
                if not tokens_lower:
                    tokens_lower = tokens

            start = to_global_frame_idx(segment.get('start', 0), n_frames)
            end = to_global_frame_idx(segment.get('end', n_frames - 1), n_frames)

            merged_labels.append(
                {
                    'id': sent_id,
                    'start': start,
                    'end': end,
                    'text': text,
                    'text_lower': text_lower,
                    'tokens': tokens,
                    'tokens_lower': tokens_lower,
                }
            )

        out_feat_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(out_feat_path, feats)

        with open(out_json_path, 'w', encoding='utf-8') as fout:
            json.dump(merged_labels, fout, ensure_ascii=False, indent=2)

        print(f'[OK] {tok_path} -> {out_feat_path} + {out_json_path}')